# 01B — External Baryonic Data Audit (THINGS and LITTLE THINGS)

**Project:** *Is the RAR Universal? Correlated-error-aware constraints on the scatter and acceleration scale of the radial acceleration relation in SPARC and independent HI samples*

**Purpose.** Phase 1 found 0 ready (category A) THINGS / LITTLE THINGS galaxies because the unified corpus has no baryonic velocity components outside SPARC, so the cross-survey feasibility gate did not pass. This notebook checks whether the missing components can be recovered from the **external source files actually present** in `01_Datasets/LITTLE_THINGS/` and `01_Datasets/THINGS/`, then recalculates the categories and the gate.

**What counts as a baryonic component.** The proposal's likelihood needs, at each radius of the observed rotation curve, separate **gas** and **stellar-disk** (and bulge, where applicable) velocity contributions, so that Υ★ can be sampled. The following are **not** components and are never treated as such: a total rotation curve, a dark-matter halo model curve, a baryon-subtracted ("dark matter only") curve, total gas or stellar **masses**, or halo-fit parameters.

**Independence.** This notebook does not use any Notebook 02 calculation. It reads the original external files and the Phase 1 audit tables, and writes only new files (nothing from Notebooks 01 or 02 is overwritten). No data are downloaded automatically; sources that could not be verified are listed with manual retrieval steps.

## 1. Environment Setup and Configuration

In [1]:
import sys, importlib.util, subprocess
IN_COLAB = "google.colab" in sys.modules
REQUIRED = {"numpy": "numpy", "pandas": "pandas", "astropy": "astropy"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os, re, io, json, hashlib, logging, platform, warnings, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import astropy
from astropy.io import ascii as astro_ascii

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 80)
RUN_TIMESTAMP = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Run started (UTC): {RUN_TIMESTAMP} | Colab: {IN_COLAB}")
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | astropy {astropy.__version__}")

Run started (UTC): 2026-10-22T09:18:07Z | Colab: True
Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | astropy 7.2.3


In [2]:
# =====================================================================
# CONFIGURATION — single place for paths and settings
# =====================================================================
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
PROJECT_ROOT = Path(os.environ.get("RAR_PROJECT_ROOT", str(DRIVE_ROOT / "RAR_Universality_Research")))

DATASETS_DIR = PROJECT_ROOT / "01_Datasets"
LT_DIR       = DATASETS_DIR / "LITTLE_THINGS"
THINGS_DIR   = DATASETS_DIR / "THINGS"
UNIFIED_DIR  = DATASETS_DIR / "UNIFIED_HI"
LT_FILES = {"ReadMe": LT_DIR / "ReadMe", "table1": LT_DIR / "table1.dat",
            "table2": LT_DIR / "table2.dat", "rotdmbar": LT_DIR / "rotdmbar.dat"}

PROCESSED_DIR      = PROJECT_ROOT / "03_Processed_Data"
RESULTS_TABLES_DIR = PROJECT_ROOT / "05_Results" / "Tables"
LOG_DIR            = PROCESSED_DIR / "logs"

# Phase 1 inputs (read-only)
P1 = {
    "feasibility": PROCESSED_DIR / "extension_feasibility_audit.csv",
    "duplicates":  PROCESSED_DIR / "cross_survey_duplicate_candidates.csv",
    "checklist":   PROCESSED_DIR / "external_data_checklist.csv",
    "inventory":   PROCESSED_DIR / "unified_galaxy_inventory.csv",
    "sparc":       PROCESSED_DIR / "sparc_galaxy_audit.csv",
}
UNIFIED_JSON = UNIFIED_DIR / "rotation_curve_corpus_v7.json"   # read only to compare LITTLE THINGS points with the CDS files

# New outputs (none of these names is used by Notebook 01 or 02)
OUT = {
    "manifest":   PROCESSED_DIR / "external_baryonic_file_manifest.csv",
    "things":     PROCESSED_DIR / "things_baryonic_audit.csv",
    "lt":         PROCESSED_DIR / "little_things_baryonic_audit.csv",
    "inventory":  PROCESSED_DIR / "external_baryonic_components_inventory.csv",
    "matches":    PROCESSED_DIR / "cross_survey_matches_updated.csv",
    "feas":       PROCESSED_DIR / "extension_feasibility_audit_updated.csv",
    "missing":    PROCESSED_DIR / "external_data_missing_items.csv",
    "registry":   PROCESSED_DIR / "external_data_source_registry.csv",
    "lt_columns": PROCESSED_DIR / "little_things_cds_column_definitions.csv",
    "lt_rc":      PROCESSED_DIR / "little_things_rotation_curves_physical_units.csv",
    "checks":     PROCESSED_DIR / "01B_integrity_checks.csv",
}
REPORT_PATH = RESULTS_TABLES_DIR / "External_Baryonic_Feasibility_Report.md"
LOG_PATH = LOG_DIR / "01B_external_baryonic_audit.log"

# Settings fixed by the final proposal
GATE_MIN_INDEPENDENT = 15
EXTENSION_SURVEYS = ("THINGS", "LITTLE_THINGS")

# Audit tolerances (documented choices)
POS_TOL_ARCSEC = 30.0           # LITTLE THINGS table1 vs corpus coordinates
MATCH_TOL_R_REL = 0.01          # corpus vs CDS radius agreement (relative)
MATCH_TOL_V_KMS = 0.5           # corpus vs CDS velocity agreement (km/s)

# Sources consulted by the notebook author on 2026-10-22 (see registry section for what each provides)
SOURCES = {
    "LT_CDS":     "https://cdsarc.cds.unistra.fr/viz-bin/cat/J/AJ/149/180",
    "LT_README":  "https://cdsarc.cds.unistra.fr/viz-bin/ReadMe/J/AJ/149/180?format=html&tex=true",
    "LT_PAPER":   "https://arxiv.org/abs/1502.01281",
    "THINGS_RC":  "https://arxiv.org/abs/0810.2100",
    "THINGS_SURVEY_VIZIER": "https://dc.g-vo.org/rr/q/lp/custom/CDS.VizieR/J/AJ/136/2563",
    "THINGS_SURVEY_PAPER": "https://arxiv.org/abs/0810.2125",
    "SPARC":      "https://astroweb.case.edu/SPARC/",
    "CORPUS":     "https://zenodo.org/records/21768240",
    "GITHUB_THINGS_CANDIDATE": "https://github.com/igomezv/Reconstructing-RC-with-ANN",
}
print("Project root:", PROJECT_ROOT)

Project root: /content/drive/MyDrive/RAR_Universality_Research


## 2. Google Drive Mounting

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    if not DRIVE_ROOT.exists():
        raise FileNotFoundError(f"Drive mounted but {DRIVE_ROOT} not found; re-run and complete authorisation.")
else:
    print("Not in Colab: using PROJECT_ROOT =", PROJECT_ROOT)

Mounted at /content/drive


## 3. File Existence, Provenance and Checksums

Checks the four LITTLE THINGS files, inspects the THINGS folder (an empty folder is a valid, reported state), checks the Phase 1 inputs, and records SHA-256 checksums for every external file.

In [4]:
def _ls(p):
    return ", ".join(sorted(x.name for x in p.iterdir())[:40]) if p.is_dir() else "(folder missing)"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}")
problems = []
for k, p in LT_FILES.items():
    if not p.is_file():
        problems.append(f"- LITTLE THINGS {k}: {p} (folder contains: {_ls(LT_DIR)})")
for k, p in P1.items():
    if not p.is_file():
        problems.append(f"- Phase 1 output {k}: {p}")
for d in (PROCESSED_DIR, RESULTS_TABLES_DIR):
    if not d.is_dir():
        problems.append(f"- output folder missing: {d}")
if problems:
    raise FileNotFoundError("Required inputs are missing:\n" + "\n".join(problems))
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("rar_01b"); logger.setLevel(logging.INFO); logger.handlers.clear()
_fh = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8"); _fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(_fh); logger.addHandler(_sh)

CHECKS = []
def record_check(section, name, passed, detail="", severity="critical"):
    status = "PASS" if passed else ("FAIL" if severity == "critical" else "WARN")
    CHECKS.append({"section": section, "check": name, "status": status, "severity": severity, "detail": str(detail)})
    (logger.info if passed else (logger.error if severity == "critical" else logger.warning))(f"[{status}] {section} | {name} | {detail}")
    return bool(passed)

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# THINGS folder: may be empty or missing; both are reported, neither stops the notebook
things_files = sorted(p for p in THINGS_DIR.rglob("*") if p.is_file()) if THINGS_DIR.is_dir() else []
THINGS_FOLDER_STATE = ("missing" if not THINGS_DIR.is_dir() else "empty" if not things_files else f"{len(things_files)} files")
record_check("Files", "THINGS folder inspected", True, THINGS_FOLDER_STATE)
if THINGS_FOLDER_STATE != "empty" and things_files:
    logger.warning("THINGS folder contains files; they are inventoried below but NOT used until their format is documented and verified.")

rows = []
for k, p in LT_FILES.items():
    rows.append({"survey": "LITTLE_THINGS", "role": k, "file": p.name, "relative_path": str(p.relative_to(PROJECT_ROOT)),
                 "size_bytes": p.stat().st_size, "sha256": sha256_of(p), "status": "present",
                 "stated_origin": SOURCES["LT_CDS"], "read_by_notebook": True})
for p in things_files:
    rows.append({"survey": "THINGS", "role": "unverified user file", "file": p.name, "relative_path": str(p.relative_to(PROJECT_ROOT)),
                 "size_bytes": p.stat().st_size, "sha256": sha256_of(p), "status": "present, format not verified",
                 "stated_origin": "unknown (record the source before use)", "read_by_notebook": False})
if not things_files:
    rows.append({"survey": "THINGS", "role": "folder", "file": "(none)", "relative_path": str(THINGS_DIR.relative_to(PROJECT_ROOT)),
                 "size_bytes": 0, "sha256": "", "status": f"THINGS folder {THINGS_FOLDER_STATE}", "stated_origin": "",
                 "read_by_notebook": False})
for k, p in P1.items():
    rows.append({"survey": "PHASE1", "role": k, "file": p.name, "relative_path": str(p.relative_to(PROJECT_ROOT)),
                 "size_bytes": p.stat().st_size, "sha256": sha256_of(p), "status": "present (read-only input)",
                 "stated_origin": "01_Data_Audit.ipynb", "read_by_notebook": True})
file_manifest = pd.DataFrame(rows)
display(file_manifest[["survey", "role", "file", "size_bytes", "status", "sha256"]])

INFO: [PASS] Files | THINGS folder inspected | empty


INFO:rar_01b:[PASS] Files | THINGS folder inspected | empty


,survey,role,file,size_bytes,status,sha256
0,LITTLE_THINGS,ReadMe,ReadMe,20246,present,c9cafe20d4096448078e9191534a7d6890262669ca9a90e802abf83dc5607c0c
1,LITTLE_THINGS,table1,table1.dat,2756,present,4dc9f3727818062ea1e424e10dcfedf385af5723c79ff3b213702cc39d96b3f4
2,LITTLE_THINGS,table2,table2.dat,4471,present,cc8ab9e5351d60b5ee017a61d0d6a1fa42d0d85a278a6c4ea136cfd53f940efb
3,LITTLE_THINGS,rotdmbar,rotdmbar.dat,109824,present,1ce3e9ee850b0a25c5f97fa36db3b0705fb558979f1e5df77eea71c3c56efe4c
4,THINGS,folder,(none),0,THINGS folder empty,
5,PHASE1,feasibility,extension_feasibility_audit.csv,84610,present (read-only input),fb3b1a3dafb95b3df70301b4900126a834c94e2059a9e1658ac95438f1b5ac41
6,PHASE1,duplicates,cross_survey_duplicate_candidates.csv,5712,present (read-only input),5557b56732c744de5aa4168b4b682e56d89c8b69e3f0ebab892c847fc2687383
7,PHASE1,checklist,external_data_checklist.csv,37896,present (read-only input),cac0462dbf9ca3f8cc10709a8bc2b2bfef4eea750915782c0e66d7b135c17501
8,PHASE1,inventory,unified_galaxy_inventory.csv,357281,present (read-only input),1f70e57626946a833bdf3a4105aced0d6517458828a98ea5360b60200608f008
9,PHASE1,sparc,sparc_galaxy_audit.csv,42792,present (read-only input),e9e877d4dd3786bb14ce2707ff0c5d82a5f2f59c7a69f3f397cdf75115d6e992


## 4. LITTLE THINGS: Parse the Official CDS ReadMe

The ReadMe is parsed for the file summary (record length, number of records) and the byte-by-byte description of each data file (bytes, format, units, label, explanation, notes). These definitions are used to read the data and are exported to `little_things_cds_column_definitions.csv`.

In [5]:
readme_text = LT_FILES["ReadMe"].read_text(encoding="latin-1")
rlines = readme_text.splitlines()
print(rlines[0] if rlines else "(empty ReadMe)")

# File summary
file_summary = []
in_summary = False
for l in rlines:
    if re.match(r"^\s*File Summary", l, re.I):
        in_summary = True; continue
    if in_summary:
        m = re.match(r"^\s*(\S+)\s+(\d+)\s+(\d+|\.)\s+(.*)$", l)
        if m and m.group(1).lower() not in ("filename",):
            file_summary.append({"file": m.group(1), "lrecl": int(m.group(2)),
                                 "records": (int(m.group(3)) if m.group(3) != "." else None), "explanation": m.group(4).strip()})
        if re.match(r"^\s*Byte-by-byte", l):
            break
file_summary = pd.DataFrame(file_summary).drop_duplicates("file")
print("Files described in the ReadMe:")
display(file_summary)

# Byte-by-byte sections
SPEC_RE = re.compile(r"^\s*(\d+)(?:\s*-\s*(\d+))?\s+([AIFE]\d+(?:\.\d+)?)\s+(\S+)\s+(\S+)\s*(.*)$")
def parse_bbb(lines):
    sections, cur, notes, cur_note, state = {}, None, {}, None, None
    for l in lines:
        m = re.match(r"^\s*Byte-by-byte Description of file:\s*(.+?)\s*$", l)
        if m:
            files = [f.strip() for f in re.split(r"[,\s]+", m.group(1)) if f.strip()]
            cur = {"files": files, "specs": [], "notes": {}}
            for f in files:
                sections[f] = cur
            state, cur_note = "specs", None
            continue
        if cur is None:
            continue
        nm = re.match(r"^\s*Note\s*\((\d+)\)\s*:?\s*(.*)$", l)
        if nm:
            cur_note = nm.group(1); cur["notes"][cur_note] = nm.group(2).strip(); state = "notes"; continue
        if state == "notes":
            if re.match(r"^\s*-{10,}\s*$", l) or re.match(r"^\s*={10,}\s*$", l):
                cur_note = None; continue
            if l.strip() and cur_note:
                cur["notes"][cur_note] += " " + l.strip()
            continue
        sm = SPEC_RE.match(l)
        if sm:
            s, e = int(sm.group(1)), int(sm.group(2) or sm.group(1))
            cur["specs"].append({"start_byte": s, "end_byte": e, "format": sm.group(3), "units": sm.group(4),
                                 "label": sm.group(5), "explanation": sm.group(6).strip()})
        elif cur["specs"] and l.startswith(" " * 20) and l.strip() and not l.strip().startswith("-"):
            cur["specs"][-1]["explanation"] += " " + l.strip()      # continuation line
    return sections

bbb = parse_bbb(rlines)
col_defs = []
for f, sec in bbb.items():
    for s in sec["specs"]:
        col_defs.append({"file": f, **s})
    for k, v in sec["notes"].items():
        col_defs.append({"file": f, "start_byte": None, "end_byte": None, "format": "", "units": "", "label": f"Note ({k})", "explanation": v})
col_defs = pd.DataFrame(col_defs)
for f in ("table1.dat", "table2.dat", "rotdmbar.dat"):
    record_check("LT ReadMe", f"byte-by-byte description found for {f}", f in bbb and len(bbb[f]["specs"]) > 0,
                 f"{len(bbb.get(f, {}).get('specs', []))} columns")
if not all(f in bbb for f in ("table1.dat", "table2.dat", "rotdmbar.dat")):
    raise ValueError("The ReadMe lacks a byte-by-byte description for one of the three data files.")
for f in ("table1.dat", "table2.dat", "rotdmbar.dat"):
    print(f"\n=== {f} ===")
    display(pd.DataFrame(bbb[f]["specs"]))
    for k, v in bbb[f]["notes"].items():
        print(f"  Note ({k}): {v}")

J/AJ/149/180  LITTLE THINGS dwarf galaxies mass models & properties  (Oh+, 2015)
Files described in the ReadMe:


,file,lrecl,records,explanation
0,ReadMe,80,NaN,This file
1,table1.dat,105,26.0,Properties and tilted-ring parameters of the
2,table2.dat,171,26.0,Mass modeling results of the LITTLE THINGS sample
3,rotdmbar.dat,63,1716.0,Rotation curves (DM + baryons) of the sample
4,dendmbar.dat,42,1624.0,Density profiles (DM + baryons) of the sample
5,rotdm.dat,63,1485.0,Rotation curves (DM only) of the sample galaxies
6,dendm.dat,42,1382.0,Density profiles (DM only) of the sample galaxies
7,nfwac.dat,38,882.0,The CDM NFW dark matter model rotation curves
8,nfwbd.dat,38,875.0,The CDM NFW dark matter model density profiles


INFO: [PASS] LT ReadMe | byte-by-byte description found for table1.dat | 22 columns


INFO:rar_01b:[PASS] LT ReadMe | byte-by-byte description found for table1.dat | 22 columns


INFO: [PASS] LT ReadMe | byte-by-byte description found for table2.dat | 29 columns


...

  Note (1): Derived from the tilted-ring analysis described in Section 3.1.1.
  Note (2): Derived from the tilted-ring analysis described in Section 3.1.1. PA is the angle measured counter-clockwise from the north direction in the sky to the major axis of the receding half of the galaxy.

=== table2.dat ===


,start_byte,end_byte,format,units,label,explanation
0,1,8,A8,---,Name,Main name of the source
1,10,13,F4.2,kpc,Rmax,[0.31/9.81] Radius where the outermost part of the rotation curve is measured
2,15,18,F4.2,kpc,R0.3,[0.2/5.43] Radius where the logarithmic slope of the total rotation curve dl...
3,20,24,F5.1,km/s,V(Rmax),[12/125.5] Rotation velocity (asymmetric drift corrected) at Rmax
4,26,30,F5.1,km/s,Viso(Rmax),[11.3/130.2] Rotation velocity of the best fitted pseudo-isothermal halo mod...
5,32,36,F5.1,---,Rmax/HI,[6/103.8] Ratio of Rmax to H_I_ beam size
6,38,41,F4.2,kpc,z0,[0.07/0.48]? Vertical scale height of the stellar disk
7,43,46,F4.1,---,c,[-0.4/25] Concentration parameter
8,48,53,F6.1,---,e_c,[0.2/1503.4]? c uncertainty (1)
9,55,58,F4.1,---,cM07,[8.1/18.6] Concentration parameter derived using an empirical relationship b...


  Note (1): A null value means that the error is unphysically large.
  Note (2): NFW model from Navarro+1995 (1995MNRAS.275...56N) & Navarro+1996b (1996ApJ...462..563N). See Section 3.4.
  Note (3): Measured in Section 4.2.
  Note (4): Derived in Section 3.2.
  Note (5): Derived in Section 3.3.
  Note (6): Using Equation 3 from Oh+2011a (2011AJ....142...24O). See Section 3.4.

=== rotdmbar.dat ===


,start_byte,end_byte,format,units,label,explanation
0,1,8,A8,---,Name,Galaxy identifier
1,10,14,A5,---,Type,Data type (1)
2,16,23,F8.6,kpc,R0.3,Radius where the logarithmic slope of the total rotation curve dlog(V)/dlog(...
3,25,34,F10.6,km/s,V0.3,Rotation velocity at R0.3
4,36,44,F9.6,---,R,Radius scaled by R0.3
5,46,54,F9.6,---,V,Velocity scaled by V0.3
6,56,63,F8.6,---,e_V,Uncertainty in V


  Note (1): Data type as follows: Data = The scaled total rotation velocity which is corrected for the asymmetric drift. Model = Pseudo-isothermal halo model fit to the total rotation velocity.


## 5. LITTLE THINGS: Read the Data Files

Each table is read twice: with **Astropy's documented CDS reader** (using the ReadMe), and with an **independent fixed-width reader** built from the parsed byte positions. The two must agree. Line lengths and record counts are checked against the ReadMe. Blank fields are missing values (CDS convention); nothing is filled in.

In [6]:
def read_fixed(fname):
    specs = pd.DataFrame(bbb[fname]["specs"])
    raw = pd.read_fwf(LT_DIR / fname, colspecs=[(s - 1, e) for s, e in zip(specs.start_byte, specs.end_byte)],
                      names=list(specs.label), dtype=str, header=None, encoding="latin-1")
    out = pd.DataFrame(index=raw.index)
    for _, s in specs.iterrows():
        txt = raw[s.label].astype("string").str.strip().replace("", pd.NA)
        out[s.label] = txt if s.format[0] == "A" else pd.to_numeric(txt, errors="coerce")
    return out, raw

def read_astropy(fname):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        t = astro_ascii.read(str(LT_DIR / fname), format="cds", readme=str(LT_FILES["ReadMe"]))
    df = t.to_pandas()
    return df, [str(x.message)[:200] for x in w], t

LT = {}
for key, fname in (("table1", "table1.dat"), ("table2", "table2.dat"), ("rotdmbar", "rotdmbar.dat")):
    lines = (LT_DIR / fname).read_text(encoding="latin-1").splitlines()
    lines = [l for l in lines if l.strip()]
    summ = file_summary[file_summary.file == fname]
    lrecl = int(summ.lrecl.iloc[0]) if len(summ) else None
    nrec = summ.records.iloc[0] if len(summ) else None
    record_check("LT files", f"{fname}: record count matches ReadMe", nrec is None or len(lines) == nrec, f"{len(lines)} lines vs ReadMe {nrec}")
    record_check("LT files", f"{fname}: no line longer than Lrecl", lrecl is None or max(len(l) for l in lines) <= lrecl,
                 f"max {max(len(l) for l in lines)} vs Lrecl {lrecl}")
    seps = sorted({c for l in lines for c in set(l) if not c.isalnum() and c not in " .+-_"})
    if seps:
        print(f"{fname}: non-alphanumeric characters present: {seps} (e.g. '|' in separator bytes is harmless if byte positions hold)")
    fw, raw = read_fixed(fname)
    try:
        ap, ap_warn, ap_table = read_astropy(fname)
        astropy_ok = True
    except Exception as exc:
        ap, ap_warn, ap_table, astropy_ok = None, [str(exc)], None, False
    record_check("LT files", f"{fname}: Astropy CDS reader succeeded", astropy_ok, ap_warn[:2] if ap_warn else "ok", severity="warning")
    if astropy_ok:
        diffs = []
        for c in fw.columns:
            if c not in ap.columns:
                diffs.append(f"{c}: missing in astropy"); continue
            a, b = fw[c], ap[c]
            if pd.api.types.is_numeric_dtype(a):
                bnum = pd.to_numeric(b, errors="coerce")
                bad = ~((a.isna() & bnum.isna()) | (np.abs(a - bnum) <= 1e-9 * np.maximum(1, np.abs(a))))
            else:
                bad = a.fillna("").astype(str).str.strip() != b.fillna("").astype(str).str.strip()
            if bad.any():
                diffs.append(f"{c}: {int(bad.sum())} rows differ")
        record_check("LT files", f"{fname}: Astropy and independent reader agree", not diffs, diffs[:5] or "identical")
    # Units recorded by astropy (if available)
    units = {c.name: str(c.unit) for c in ap_table.columns.values()} if astropy_ok else {}
    LT[key] = {"df": fw, "astropy_units": units, "n_lines": len(lines)}
    print(f"{fname}: {len(fw)} rows x {fw.shape[1]} columns; missing values per column:")
    display(fw.isna().sum().to_frame("n_missing").T)

INFO: [PASS] LT files | table1.dat: record count matches ReadMe | 26 lines vs ReadMe 26.0


INFO:rar_01b:[PASS] LT files | table1.dat: record count matches ReadMe | 26 lines vs ReadMe 26.0


INFO: [PASS] LT files | table1.dat: no line longer than Lrecl | max 105 vs Lrecl 105


INFO:rar_01b:[PASS] LT files | table1.dat: no line longer than Lrecl | max 105 vs Lrecl 105


table1.dat: non-alphanumeric characters present: ['|'] (e.g. '|' in separator bytes is harmless if byte positions hold)
INFO: [PASS] LT files | table1.dat: Astropy CDS reader succeeded | ok


...

INFO:rar_01b:[PASS] LT files | rotdmbar.dat: Astropy CDS reader succeeded | ok


INFO: [PASS] LT files | rotdmbar.dat: Astropy and independent reader agree | identical


INFO:rar_01b:[PASS] LT files | rotdmbar.dat: Astropy and independent reader agree | identical


rotdmbar.dat: 1716 rows x 7 columns; missing values per column:


,Name,Type,R0.3,V0.3,R,V,e_V
n_missing,0,0,0,0,0,0,0


In [7]:
# Resolve the columns this audit relies on (alternative spellings accepted), check their documented units,
# and rename them to canonical names. Labels and units come from the parsed ReadMe.
LABEL_ALIASES = {
    "table1.dat": {"Name": ["Name"], "Dist": ["Dist", "D"], "i": ["i", "Inc", "incl"], "e_i": ["e_i", "e_Inc"],
                   "RAh": ["RAh"], "RAm": ["RAm"], "RAs": ["RAs"], "DE-": ["DE-"], "DEd": ["DEd"], "DEm": ["DEm"], "DEs": ["DEs"]},
    "table2.dat": {"Name": ["Name"], "Rmax": ["Rmax"], "R0.3": ["R0.3", "R03"],
                   "Mgas": ["Mgas"], "MstarK": ["MstarK", "M*K"], "MstarSED": ["MstarSED", "M*SED"]},
    "rotdmbar.dat": {"Name": ["Name"], "Type": ["Type"], "R0.3": ["R0.3", "R03"], "V0.3": ["V0.3", "V03"],
                     "R": ["R"], "V": ["V"], "e_V": ["e_V"]},
}
REQUIRED_UNITS = {("table1.dat", "Dist"): "Mpc", ("table1.dat", "i"): "deg", ("table1.dat", "e_i"): "deg",
                  ("table2.dat", "Rmax"): "kpc", ("table2.dat", "R0.3"): "kpc",
                  ("rotdmbar.dat", "R0.3"): "kpc", ("rotdmbar.dat", "V0.3"): "km/s",
                  ("rotdmbar.dat", "R"): "---", ("rotdmbar.dat", "V"): "---", ("rotdmbar.dat", "e_V"): "---"}
OPTIONAL = {("table2.dat", "Mgas"), ("table2.dat", "MstarK"), ("table2.dat", "MstarSED")}
specs_by = {f: {x["label"]: x for x in bbb[f]["specs"]} for f in LABEL_ALIASES}
RESOLVED, unresolved = {}, []
for f, amap in LABEL_ALIASES.items():
    for canon, alts in amap.items():
        hit = next((a for a in alts if a in specs_by[f]), None)
        if hit is None and (f, canon) not in OPTIONAL:
            unresolved.append((f, canon))
        RESOLVED[(f, canon)] = hit
if unresolved:
    raise KeyError(f"Required LITTLE THINGS columns not found in the ReadMe: {unresolved}. "
                   f"Labels present: { {f: list(v) for f, v in specs_by.items()} }")
for (f, canon), unit in REQUIRED_UNITS.items():
    lab = RESOLVED[(f, canon)]
    record_check("LT units", f"{f}: {canon} documented in {unit}", specs_by[f][lab]["units"] == unit,
                 f"label '{lab}', ReadMe unit '{specs_by[f][lab]['units']}'")
key_of = {"table1.dat": "table1", "table2.dat": "table2", "rotdmbar.dat": "rotdmbar"}
for f, amap in LABEL_ALIASES.items():
    ren = {RESOLVED[(f, c)]: c for c in amap if RESOLVED[(f, c)]}
    LT[key_of[f]]["df"] = LT[key_of[f]]["df"].rename(columns=ren)
    for c in amap:
        if c not in LT[key_of[f]]["df"]:
            LT[key_of[f]]["df"][c] = np.nan        # optional column absent: stays missing, never filled

def norm_name(n):
    s = str(n).upper().strip()
    s = re.sub(r"[\s_\-\.]+", "", s)
    return re.sub(r"(?<![0-9])0+(?=[0-9])", "", s)

t1, t2, rc = LT["table1"]["df"].copy(), LT["table2"]["df"].copy(), LT["rotdmbar"]["df"].copy()
for t in (t1, t2, rc):
    t["norm"] = t["Name"].map(norm_name)
record_check("LT IDs", "table1 names unique", not t1.norm.duplicated().any(), t1.Name[t1.norm.duplicated()].tolist() or f"{t1.norm.nunique()} galaxies")
record_check("LT IDs", "table2 names unique", not t2.norm.duplicated().any(), f"{t2.norm.nunique()} galaxies")
record_check("LT IDs", "table1 and table2 list the same galaxies", set(t1.norm) == set(t2.norm),
             f"t1 only {sorted(set(t1.norm)-set(t2.norm))}; t2 only {sorted(set(t2.norm)-set(t1.norm))}")
record_check("LT IDs", "every rotdmbar galaxy is in table1", set(rc.norm) <= set(t1.norm), sorted(set(rc.norm) - set(t1.norm)))
record_check("LT IDs", "every table1 galaxy has rotdmbar rows", set(t1.norm) <= set(rc.norm), sorted(set(t1.norm) - set(rc.norm)), severity="warning")

# Coordinates (degrees) from sexagesimal columns
sign = np.where(t1["DE-"].fillna("+").astype(str).str.strip() == "-", -1.0, 1.0)
t1["ra_deg"] = 15.0 * (t1.RAh + t1.RAm / 60 + t1.RAs / 3600)
t1["dec_deg"] = sign * (t1.DEd + t1.DEm / 60 + t1.DEs / 3600)
record_check("LT values", "distances positive", bool((t1.Dist > 0).all()), f"range {t1.Dist.min()}–{t1.Dist.max()} Mpc")
record_check("LT values", "inclinations within [0, 90] deg", bool(t1.i.between(0, 90).all()), f"range {t1.i.min()}–{t1.i.max()}")
small_ei = t1.loc[t1.e_i < 1.0, "Name"].tolist()
record_check("LT values", "inclination uncertainties >= 1 deg", not small_ei,
             f"{len(small_ei)} galaxies with e_i < 1 deg (likely fixed in the tilted-ring fit; not a realistic error): {small_ei}", severity="warning")

INFO: [PASS] LT units | table1.dat: Dist documented in Mpc | label 'Dist', ReadMe unit 'Mpc'


INFO:rar_01b:[PASS] LT units | table1.dat: Dist documented in Mpc | label 'Dist', ReadMe unit 'Mpc'


INFO: [PASS] LT units | table1.dat: i documented in deg | label 'i', ReadMe unit 'deg'


INFO:rar_01b:[PASS] LT units | table1.dat: i documented in deg | label 'i', ReadMe unit 'deg'


INFO: [PASS] LT units | table1.dat: e_i documented in deg | label 'e_i', ReadMe unit 'deg'


...

INFO: [PASS] LT values | distances positive | range 0.7–10.3 Mpc


INFO:rar_01b:[PASS] LT values | distances positive | range 0.7–10.3 Mpc


INFO: [PASS] LT values | inclinations within [0, 90] deg | range 22.6–82.5


INFO:rar_01b:[PASS] LT values | inclinations within [0, 90] deg | range 22.6–82.5


False

## 6. LITTLE THINGS: What Each File Contains

Determined from the ReadMe definitions **and** the data themselves:

* `rotdmbar.dat`: the `Type` column separates `Data` rows (scaled observed total rotation curve) from `Model` rows (a fitted halo model). Velocities and radii are **dimensionless**, scaled by each galaxy's `V0.3` and `R0.3`. Physical values are R = R·R0.3 (kpc), V = V·V0.3 (km/s), σ_V = e_V·V0.3.
* The scaling interpretation is **tested**: R0.3 must equal table2's R0.3; V(R = 1) should be close to 1 for Data rows; and the largest physical radius should be close to table2's Rmax.
* Every file's labels and explanations are scanned for gas, stellar-disk, bulge or baryonic **velocity profile** columns.

In [8]:
types = rc["Type"].astype(str).str.strip()
print("rotdmbar Type values:", types.value_counts().to_dict())
type_notes = " ".join(bbb["rotdmbar.dat"]["notes"].values())
print("ReadMe notes for rotdmbar.dat:", type_notes[:600])
record_check("LT content", "rotdmbar Type values are documented in the ReadMe notes",
             all(t in type_notes for t in types.unique()), f"types {sorted(types.unique())}", severity="warning")

# Physical units (DERIVED by documented scaling; original scaled values kept alongside)
rc["R_kpc"] = rc["R"] * rc["R0.3"]
rc["V_kms"] = rc["V"] * rc["V0.3"]
rc["eV_kms"] = rc["e_V"] * rc["V0.3"]
rc["row_type"] = types
data_rows = rc[rc.row_type.str.lower() == "data"].copy()
model_rows = rc[rc.row_type.str.lower() != "data"].copy()

# Tests of the scaling interpretation
r03 = data_rows.groupby("norm")["R0.3"].agg(["min", "max"])
record_check("LT scaling", "R0.3 constant within each galaxy", bool((r03["max"] - r03["min"]).abs().max() < 1e-6), "ok")
cmp_r03 = r03["min"].rename("R03_rotdmbar").to_frame().join(t2.set_index("norm")[["R0.3", "Rmax"]])
cmp_r03["absdiff"] = (cmp_r03.R03_rotdmbar - cmp_r03["R0.3"]).abs()
record_check("LT scaling", "rotdmbar R0.3 equals table2 R0.3 (to table precision 0.01 kpc)",
             bool((cmp_r03.absdiff.dropna() <= 0.006).all()), f"max |diff| {cmp_r03.absdiff.max():.4f} kpc", severity="warning")
def v_at_r1(g):
    g = g.groupby("R")["V"].mean().sort_index()
    return float(np.interp(1.0, g.index.values, g.values)) if g.index.min() <= 1.0 <= g.index.max() else np.nan
v1 = data_rows.groupby("norm").apply(v_at_r1)
record_check("LT scaling", "scaled Data velocity at R = 1 is ~1 (median |V(1) - 1| < 0.1)",
             bool(np.nanmedian(np.abs(v1 - 1)) < 0.1), f"median |V(1)-1| = {np.nanmedian(np.abs(v1 - 1)):.3f}", severity="warning")
rmax = data_rows.groupby("norm")["R_kpc"].max().rename("Rmax_from_data").to_frame().join(t2.set_index("norm")["Rmax"])
rmax["rel_diff"] = (rmax.Rmax_from_data - rmax.Rmax).abs() / rmax.Rmax
record_check("LT scaling", "largest Data radius matches table2 Rmax within 10% for most galaxies",
             bool((rmax.rel_diff < 0.10).mean() >= 0.8), f"median rel. diff {rmax.rel_diff.median():.3f}", severity="warning")
record_check("LT content", "Data rows have positive uncertainties", bool((data_rows.e_V > 0).all()),
             f"{int((~(data_rows.e_V > 0)).sum())} rows without positive e_V", severity="warning")

# Scan every documented column for baryonic velocity-profile content
COMPONENT_PAT = re.compile(r"\b(gas|HI|stellar|stars?|disk|disc|bulge|baryon\w*|Vgas|Vdisk|Vbul|Vstar|Vbar)\b", re.I)
scan = col_defs[col_defs.start_byte.notna()].copy()
scan["mentions_baryons"] = scan.apply(lambda r: bool(COMPONENT_PAT.search(f"{r.label} {r.explanation}")), axis=1)
scan["velocity_unit"] = scan.units.str.contains("km/s", regex=False)
scan["radial_profile_file"] = scan.file.isin(["rotdmbar.dat"]) | scan.file.str.contains(r"rot|vel|curve", case=False, regex=True)
scan["is_component_profile_candidate"] = scan.mentions_baryons & scan.velocity_unit & scan.radial_profile_file
display(scan[scan.mentions_baryons][["file", "label", "units", "explanation", "velocity_unit", "is_component_profile_candidate"]])
component_candidates = scan[scan.is_component_profile_candidate]
record_check("LT content", "no gas/stellar velocity-profile columns in the LITTLE THINGS files read",
             component_candidates.empty, component_candidates[["file", "label"]].values.tolist() or "none found", severity="warning")
mass_cols = scan[scan.mentions_baryons & ~scan.velocity_unit & (scan.file == "table2.dat")]
print("Baryonic quantities in table2.dat (global values, NOT radial profiles):", mass_cols.label.tolist())

# Other files listed in the ReadMe but not present locally
not_local = [f for f in file_summary.file if f.lower() != "readme" and not (LT_DIR / f).exists()]
print("Files described in the ReadMe but not in LITTLE_THINGS/:", not_local)
for f in not_local:
    exp = file_summary.loc[file_summary.file == f, "explanation"].iloc[0]
    print(f"   {f}: {exp}")

rotdmbar Type values: {'Data': 877, 'Model': 839}
ReadMe notes for rotdmbar.dat: Data type as follows: Data = The scaled total rotation velocity which is corrected for the asymmetric drift. Model = Pseudo-isothermal halo model fit to the total rotation velocity.
INFO: [PASS] LT content | rotdmbar Type values are documented in the ReadMe notes | types ['Data', 'Model']


INFO:rar_01b:[PASS] LT content | rotdmbar Type values are documented in the ReadMe notes | types ['Data', 'Model']


INFO: [PASS] LT scaling | R0.3 constant within each galaxy | ok


INFO:rar_01b:[PASS] LT scaling | R0.3 constant within each galaxy | ok


INFO: [PASS] LT scaling | scaled Data velocity at R = 1 is ~1 (median |V(1) - 1| < 0.1) | median |V(1)-1| = 0.000


...

INFO:rar_01b:[PASS] LT scaling | largest Data radius matches table2 Rmax within 10% for most galaxies | median rel. diff 0.000


,file,label,units,explanation,velocity_unit,is_component_profile_candidate
18,table1.dat,log(SFR)Ha,Msun/yr/kpc2,[-4.1/0.19]? H{alpha} star formation rates normalized to the area ({pi}R^2^ ...,False,False
20,table1.dat,log(SFR)FUV,Msun/yr/kpc2,[-3.21/-0.01]? GALEX FUV star formation rates normalized to the area ({pi}R^...,False,False
29,table2.dat,Rmax/HI,---,[6/103.8] Ratio of Rmax to H_I_ beam size,False,False
30,table2.dat,z0,kpc,[0.07/0.48]? Vertical scale height of the stellar disk,False,False
36,table2.dat,V200M07,km/s,"[12.4/598.2]? V200 values fitted after fixing c to cM07 assuming a ""minimum ...",True,False
42,table2.dat,alphamin,---,[-1.25/1.16] Logarithmic inner slope of the total matter (DM halo+baryons) d...,False,False
44,table2.dat,f_alphamin,---,The dynamical contribution by the stellar component is included in the DM de...,False,False
47,table2.dat,f_alpha3.6,---,The dynamical contribution by the stellar component is included in the DM de...,False,False
48,table2.dat,Mgas,10+7Msun,[0.14/132.52] Gas mass (4),False,False
49,table2.dat,MstarK,10+7Msun,[0.04/20.7]? Stellar mass derived from the kinematic analysis (5),False,False


INFO: [PASS] LT content | no gas/stellar velocity-profile columns in the LITTLE THINGS files read | none found


INFO:rar_01b:[PASS] LT content | no gas/stellar velocity-profile columns in the LITTLE THINGS files read | none found


Baryonic quantities in table2.dat (global values, NOT radial profiles): ['Rmax/HI', 'z0', 'alphamin', 'f_alphamin', 'f_alpha3.6', 'Mgas', 'MstarK', 'MstarSED']
Files described in the ReadMe but not in LITTLE_THINGS/: ['dendmbar.dat', 'rotdm.dat', 'dendm.dat', 'nfwac.dat', 'nfwbd.dat']
   dendmbar.dat: Density profiles (DM + baryons) of the sample
   rotdm.dat: Rotation curves (DM only) of the sample galaxies
   dendm.dat: Density profiles (DM only) of the sample galaxies
   nfwac.dat: The CDM NFW dark matter model rotation curves
   nfwbd.dat: The CDM NFW dark matter model density profiles


In [9]:
# Compare the corpus LITTLE THINGS points with the CDS Data and Model rows (provenance check)
corpus_cmp = []
if UNIFIED_JSON.is_file():
    corpus = json.loads(UNIFIED_JSON.read_text(encoding="utf-8"))
    gals = corpus["galaxies"] if isinstance(corpus, dict) else corpus
    for g in gals:
        if "LITTLE" not in str(g.get("survey", "")).upper():
            continue
        n = norm_name(g.get("galaxy"))
        pts = pd.DataFrame(g.get("data", []))
        vcol = next((c for c in ("Vrot", "Vobs") if c in pts.columns), None)
        if pts.empty or "Rad" not in pts or vcol is None:
            corpus_cmp.append({"norm": n, "corpus_name": g.get("galaxy"), "n_corpus": len(pts)}); continue
        d = data_rows[data_rows.norm == n]; m = model_rows[model_rows.norm == n]
        def match_mask(ref):
            if ref.empty: return np.zeros(len(pts), bool)
            R, V = ref.R_kpc.to_numpy(), ref.V_kms.to_numpy()
            return np.array([bool(((np.abs(R - r) <= MATCH_TOL_R_REL * max(abs(r), 1e-3)) & (np.abs(V - v) <= MATCH_TOL_V_KMS)).any())
                             for r, v in zip(pts.Rad.astype(float), pts[vcol].astype(float))])
        md_, mm_ = match_mask(d), match_mask(m)
        corpus_cmp.append({"norm": n, "corpus_name": g.get("galaxy"), "n_corpus": len(pts),
                           "n_cds_data_rows": len(d), "n_cds_model_rows": len(m),
                           "corpus_pts_matching_cds_data": int(md_.sum()), "corpus_pts_matching_cds_model": int(mm_.sum()),
                           "corpus_pts_matching_model_only": int((mm_ & ~md_).sum()),
                           "corpus_pts_matching_neither": int((~mm_ & ~md_).sum())})
corpus_cmp = pd.DataFrame(corpus_cmp)
if len(corpus_cmp):
    display(corpus_cmp)
    mixed = corpus_cmp[corpus_cmp.get("corpus_pts_matching_model_only", pd.Series(0, index=corpus_cmp.index)).fillna(0) > 0]
    record_check("Corpus provenance", "corpus LITTLE THINGS points contain no CDS halo-Model rows", mixed.empty,
                 f"{len(mixed)} galaxies have corpus points matching only Model rows: {mixed.corpus_name.tolist()[:10]}", severity="warning")
    if not mixed.empty:
        print("NOTE: Some corpus LITTLE THINGS 'data' points coincide with CDS 'Model' rows (fitted halo curves). "
              "If confirmed, the Phase 1 'duplicate radii' in the corpus are Data+Model pairs, not approaching/receding halves. "
              "Use the CDS Data rows, not the corpus array, for any LITTLE THINGS rotation curve.")
else:
    print("Unified corpus JSON not found or no LITTLE THINGS records; comparison skipped.")

,norm,corpus_name,n_corpus,n_cds_data_rows,n_cds_model_rows,corpus_pts_matching_cds_data,corpus_pts_matching_cds_model,corpus_pts_matching_model_only,corpus_pts_matching_neither
0,CVNIDWA,CVnIdwA,30,14,16,16,18,14,0
1,DDO210,DDO_210,16,8,8,13,13,3,0
2,DDO216,DDO_216,26,13,13,17,17,9,0
3,DDO53,DDO_53,27,14,13,19,18,8,0
4,DDO70,DDO_70,48,24,24,27,27,21,0
5,IC10,IC_10,50,25,25,30,30,20,0
6,IC1613,IC_1613,218,109,109,127,126,91,0
7,UGC8508,UGC_8508,50,25,25,34,34,16,0
8,WLM,WLM,266,140,126,203,184,63,0
9,DDO50,DDO_50,170,85,85,94,94,76,0


NOTE: Some corpus LITTLE THINGS 'data' points coincide with CDS 'Model' rows (fitted halo curves). If confirmed, the Phase 1 'duplicate radii' in the corpus are Data+Model pairs, not approaching/receding halves. Use the CDS Data rows, not the corpus array, for any LITTLE THINGS rotation curve.


## 7. LITTLE THINGS: Per-Galaxy Baryonic Audit

Status vocabulary for each required quantity:

* `verified_numeric`: present as numbers in a file read by this notebook, with documented units.
* `literature_only`: described or plotted in the original paper, but no numerical profile was found in the files read here or in the CDS file list.
* `missing`: neither.
* `not_applicable`: physically not required. Bulges are treated as not applicable only where the corpus records `has_bulge = False`; otherwise the bulge status is `missing`.

In [10]:
inv = pd.read_csv(P1["inventory"])
inv["norm"] = inv["galaxy"].map(norm_name)
def corpus_flag(norm, survey, col):
    r = inv[(inv.norm == norm) & (inv.survey == survey)]
    if r.empty or col not in r: return None
    v = r[col].iloc[0]
    return None if pd.isna(v) else str(v).strip().lower() in ("true", "1", "1.0", "yes")

def quantity_rows(galaxy, survey, statuses):
    return [{"galaxy": galaxy, "survey": survey, "quantity": q, "status": s, "source": src, "note": note}
            for q, (s, src, note) in statuses.items()]

lt_rows, inv_rows = [], []
for _, g in t1.iterrows():
    n = g.norm
    d = data_rows[data_rows.norm == n]; t2r = t2[t2.norm == n]
    has_rc = len(d) > 0
    has_err = has_rc and bool((d.e_V > 0).all())
    bulge_flag = corpus_flag(n, "LITTLE_THINGS", "json_has_bulge")
    ml_vals = {c: (t2r[c].iloc[0] if len(t2r) else np.nan) for c in ("MstarK", "MstarSED", "Mgas")}
    statuses = {
        "radius":            ("verified_numeric" if has_rc else "missing", "rotdmbar.dat (R x R0.3)", "kpc, derived from documented scaling"),
        "observed_rotation_velocity": ("verified_numeric" if has_rc else "missing", "rotdmbar.dat Type=Data (V x V0.3)", "total curve, asymmetric-drift corrected"),
        "velocity_uncertainty": ("verified_numeric" if has_err else "missing", "rotdmbar.dat e_V x V0.3", ""),
        "gas_velocity_contribution": ("literature_only", SOURCES["LT_PAPER"],
                                      "mass models in Oh et al. 2015; no gas velocity profile in the CDS files read; table2 gives total Mgas only"),
        "stellar_disk_velocity_contribution": ("literature_only", SOURCES["LT_PAPER"],
                                      "no stellar velocity profile in the CDS files read; table2 gives total stellar masses only"),
        "bulge_velocity_contribution": ("not_applicable" if bulge_flag is False else "missing",
                                        "corpus has_bulge" if bulge_flag is False else "", "dwarf galaxy; per corpus flag" if bulge_flag is False else "bulge status unknown"),
        "distance":          ("verified_numeric" if pd.notna(g.Dist) else "missing", "table1.dat Dist", "Mpc (Hunter+2012)"),
        "inclination":       ("verified_numeric" if pd.notna(g.i) else "missing", "table1.dat i, e_i", f"e_i = {g.e_i} deg"),
        "stellar_mass_to_light": ("literature_only", "table2.dat MstarK/MstarSED; Oh et al. 2015 Sec. 3.3",
                                  "only total stellar masses are tabulated; no Upsilon profile"),
        "radii_compatibility": ("not_assessable", "", "no component profiles exist to align with the observed radii"),
    }
    cat = "B" if has_rc else "C"
    reason = ("observed rotation curve and uncertainties available; gas and stellar velocity profiles not available numerically"
              if has_rc else "no Data rows in rotdmbar.dat")
    lt_rows.append({"galaxy": g.Name, "norm": n, "ra_deg": g.ra_deg, "dec_deg": g.dec_deg, "distance_mpc": g.Dist,
                    "inc_deg": g.i, "e_inc_deg": g.e_i, "n_data_rows": len(d),
                    "n_model_rows": int((model_rows.norm == n).sum()),
                    "R_max_kpc_data": d.R_kpc.max() if has_rc else np.nan, "R0.3_kpc": d["R0.3"].iloc[0] if has_rc else np.nan,
                    "V0.3_kms": d["V0.3"].iloc[0] if has_rc else np.nan,
                    "Mgas_table2": ml_vals["Mgas"], "MstarK_table2": ml_vals["MstarK"], "MstarSED_table2": ml_vals["MstarSED"],
                    **{f"status_{q}": s for q, (s, _, _) in statuses.items()},
                    "has_numeric_baryonic_decomposition": False,
                    "category_from_files": cat, "category_reason": reason})
    inv_rows += quantity_rows(g.Name, "LITTLE_THINGS", statuses)
lt_audit = pd.DataFrame(lt_rows)
display(lt_audit[["galaxy", "n_data_rows", "n_model_rows", "distance_mpc", "inc_deg", "status_gas_velocity_contribution",
                  "status_stellar_disk_velocity_contribution", "category_from_files"]])

# Physical-unit rotation curves (DERIVED from the CDS files by documented scaling; Data and Model kept apart)
lt_rc_out = rc[["Name", "norm", "row_type", "R0.3", "V0.3", "R", "V", "e_V", "R_kpc", "V_kms", "eV_kms"]].rename(columns={"Name": "galaxy"})

,galaxy,n_data_rows,n_model_rows,distance_mpc,inc_deg,status_gas_velocity_contribution,status_stellar_disk_velocity_contribution,category_from_files
0,CVnIdwA,14,16,3.6,66.5,literature_only,literature_only,B
1,DDO_43,9,0,7.8,40.6,literature_only,literature_only,B
2,DDO_46,23,19,6.1,27.9,literature_only,literature_only,B
3,DDO_47,34,34,5.2,45.5,literature_only,literature_only,B
4,DDO_50,85,85,3.4,49.7,literature_only,literature_only,B
5,DDO_52,16,16,10.3,43.0,literature_only,literature_only,B
6,DDO_53,14,13,3.6,27.0,literature_only,literature_only,B
7,DDO_70,24,24,1.3,50.0,literature_only,literature_only,B
8,DDO_87,33,33,7.7,55.5,literature_only,literature_only,B
9,DDO_101,9,9,6.4,51.0,literature_only,literature_only,B


## 8. THINGS: Folder Inspection and Per-Galaxy Audit

If `01_Datasets/THINGS/` is empty, THINGS galaxies are assessed from the Phase 1 corpus inventory (observed curves) and the literature record. Files placed in the folder are inventoried and previewed but **not** used to upgrade any galaxy until their format and provenance are documented.

In [11]:
if things_files:
    print("Files in THINGS/ (preview of first lines; not used for classification):")
    for p in things_files[:20]:
        try:
            head = p.read_text(encoding="latin-1", errors="replace").splitlines()[:5]
        except Exception as exc:
            head = [f"(unreadable as text: {exc})"]
        print(f"--- {p.name}"); print("\n".join(head))
else:
    print(f"THINGS folder state: {THINGS_FOLDER_STATE}. No external THINGS files to read.")

def tb(v):
    return str(v).strip().lower() in ("true", "1", "1.0", "yes")
th = inv[inv.survey == "THINGS"].copy()
th_rows = []
for _, g in th.iterrows():
    npts = pd.to_numeric(g.get("n_points_actual"), errors="coerce")
    has_rc = bool(pd.notna(npts) and npts > 0 and tb(g.get("has_radius")) and tb(g.get("has_velocity")))
    has_err = tb(g.get("has_velocity_error"))
    bulge_flag = corpus_flag(g.norm, "THINGS", "json_has_bulge")
    statuses = {
        "radius": ("verified_numeric" if has_rc else "missing", "unified corpus (de Blok et al. 2008 curves)", ""),
        "observed_rotation_velocity": ("verified_numeric" if has_rc else "missing", "unified corpus", ""),
        "velocity_uncertainty": ("verified_numeric" if has_err else "missing", "unified corpus", ""),
        "gas_velocity_contribution": ("literature_only" if has_rc else "missing", SOURCES["THINGS_RC"],
                                      "mass models shown in de Blok et al. 2008; no public numerical profile located"),
        "stellar_disk_velocity_contribution": ("literature_only" if has_rc else "missing", SOURCES["THINGS_RC"],
                                               "3.6 um disk models in the paper; no public numerical profile located"),
        "bulge_velocity_contribution": ("not_applicable" if bulge_flag is False else ("literature_only" if has_rc else "missing"),
                                        SOURCES["THINGS_RC"], "bulge components modelled for some THINGS galaxies in the paper"),
        "distance": ("verified_numeric" if tb(g.get("has_distance")) else "missing", "unified corpus", ""),
        "inclination": ("verified_numeric" if tb(g.get("has_inclination")) else "missing", "unified corpus", ""),
        "stellar_mass_to_light": ("literature_only", SOURCES["THINGS_RC"],
                                  "Upsilon_3.6 from J-K colours; diet-Salpeter and Kroupa variants (Tables 3-4 of the paper)"),
        "radii_compatibility": ("not_assessable", "", "no component profiles available locally"),
    }
    cat = "B" if has_rc else "C"
    th_rows.append({"galaxy": g.galaxy, "norm": g.norm, "n_points_corpus": g.get("n_points_actual"),
                    **{f"status_{q}": s for q, (s, _, _) in statuses.items()},
                    "has_numeric_baryonic_decomposition": False, "things_folder_state": THINGS_FOLDER_STATE,
                    "category_from_files": cat,
                    "category_reason": ("observed curve available; gas/stellar profiles not available numerically" if has_rc
                                        else "no rotation-curve points (THINGS datacube stub)")})
    inv_rows += quantity_rows(g.galaxy, "THINGS", statuses)
things_audit = pd.DataFrame(th_rows)
display(things_audit[["galaxy", "n_points_corpus", "status_velocity_uncertainty", "status_gas_velocity_contribution", "category_from_files"]])
components_inventory = pd.DataFrame(inv_rows)

THINGS folder state: empty. No external THINGS files to read.


,galaxy,n_points_corpus,status_velocity_uncertainty,status_gas_velocity_contribution,category_from_files
0,NGC628,0,missing,missing,C
1,NGC925,96,verified_numeric,literature_only,B
2,NGC1569,0,missing,missing,C
3,NGC2366,75,verified_numeric,literature_only,B
4,NGC2403,288,verified_numeric,literature_only,B
5,HoII,0,missing,missing,C
6,M81DwA,0,missing,missing,C
7,DDO53,0,missing,missing,C
8,NGC2841,141,verified_numeric,literature_only,B
9,NGC2903,97,verified_numeric,literature_only,B


## 9. Cross-Survey Matching (SPARC duplicates)

All Phase 1 candidate pairs are kept unchanged. Matching is then recomputed for every THINGS and LITTLE THINGS galaxy against the SPARC table using normalised names and the same alias table as Phase 1. LITTLE THINGS `table1` coordinates are matched to the corpus coordinates to validate identifiers. SPARC has no coordinates in these files, so SPARC matches are name-based.

* **Confirmed SPARC duplicate:** identical normalised name.
* **Ambiguous:** alias-only match, or flagged for manual review in Phase 1. These are not counted as independent.

In [12]:
ALIAS_GROUPS = [
    ["HOII", "HOLMBERGII", "DDO50", "UGC4305"], ["HOI", "HOLMBERGI", "DDO63", "UGC5139"],
    ["IC2574", "DDO81", "UGC5666"], ["NGC2366", "DDO42", "UGC3851"], ["NGC4214", "UGC7278"],
    ["DDO154", "UGC8024"], ["DDO168", "UGC8320"], ["DDO126", "UGC7559"], ["DDO87", "UGC5918"],
    ["DDO133", "UGC7698"], ["DDO216", "UGC12613", "PEGASUSDWARF"], ["DDO53", "UGC4459"],
    ["DDO70", "UGC5373", "SEXTANSB"], ["CVNIDWA", "UGCA292"], ["WLM", "DDO221"],
    ["NGC3031", "M81"], ["NGC5194", "M51"], ["NGC5236", "M83"], ["NGC5457", "M101"],
]
ALIAS_OF = {a: i for i, grp in enumerate(ALIAS_GROUPS) for a in grp}

sparc = pd.read_csv(P1["sparc"]); sparc["norm"] = sparc["galaxy"].map(norm_name)
sparc_by_norm = dict(zip(sparc.norm, sparc.galaxy))
sparc_by_alias = {}
for n, gname in sparc_by_norm.items():
    if n in ALIAS_OF: sparc_by_alias.setdefault(ALIAS_OF[n], []).append(gname)
sparc_eligible = set(sparc.loc[sparc.get("passes_quality_cuts", pd.Series(True, index=sparc.index)).astype(bool), "galaxy"])

p1_dups = pd.read_csv(P1["duplicates"])
p1_flag = {}
if not p1_dups.empty and "galaxy_a" in p1_dups:
    for r in p1_dups.itertuples():
        for (ga, sa, gb, sb) in ((r.galaxy_a, r.survey_a, r.galaxy_b, r.survey_b), (r.galaxy_b, r.survey_b, r.galaxy_a, r.survey_a)):
            if sa in EXTENSION_SURVEYS and sb in ("SPARC", "SPARC_MRT"):
                p1_flag.setdefault((norm_name(ga), sa), []).append((gb, bool(getattr(r, "needs_manual_review", False)), r.match_methods))

ext_records = pd.concat([
    pd.DataFrame({"galaxy": lt_audit.galaxy, "norm": lt_audit.norm, "survey": "LITTLE_THINGS", "origin": "CDS table1"}),
    pd.DataFrame({"galaxy": things_audit.galaxy, "norm": things_audit.norm, "survey": "THINGS", "origin": "corpus inventory"}),
    pd.DataFrame({"galaxy": inv.loc[inv.survey == "LITTLE_THINGS", "galaxy"], "norm": inv.loc[inv.survey == "LITTLE_THINGS", "norm"],
                  "survey": "LITTLE_THINGS", "origin": "corpus inventory"}),
]).drop_duplicates(["norm", "survey"], keep="first")

new_rows, dup_status = [], {}
for r in ext_records.itertuples():
    exact = sparc_by_norm.get(r.norm)
    alias = [x for x in sparc_by_alias.get(ALIAS_OF.get(r.norm, -1), []) if norm_name(x) != r.norm]
    p1 = p1_flag.get((r.norm, r.survey), [])
    p1_review = any(rv for _, rv, _ in p1)
    if exact:
        status = "confirmed_sparc_duplicate"
    elif alias or p1:
        status = "ambiguous_possible_sparc_duplicate"
    else:
        status = "independent"
    if exact and p1_review:
        status = "confirmed_name_match_flagged_in_phase1"
    dup_status[(r.norm, r.survey)] = status
    partners = sorted(set(([exact] if exact else []) + alias + [p for p, _, _ in p1]))
    new_rows.append({"galaxy": r.galaxy, "survey": r.survey, "norm": r.norm, "record_origin": r.origin,
                     "sparc_partner": "; ".join(partners), "exact_name_match": bool(exact), "alias_match": bool(alias),
                     "found_in_phase1": bool(p1), "phase1_needs_manual_review": p1_review,
                     "sparc_partner_in_eligible_153": any(p in sparc_eligible for p in partners),
                     "duplicate_status": status, "counts_as_independent": status == "independent"})
matches_new = pd.DataFrame(new_rows)

# Identifier validation: LITTLE THINGS table1 vs corpus coordinates
inv_lt = inv[inv.survey == "LITTLE_THINGS"]
def sep_arcsec(ra1, d1, ra2, d2):
    ra1, d1, ra2, d2 = map(np.radians, (ra1, d1, ra2, d2))
    h = np.sin((d2 - d1) / 2) ** 2 + np.cos(d1) * np.cos(d2) * np.sin((ra2 - ra1) / 2) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))) * 3600
pos_rows = []
for r in lt_audit.itertuples():
    c = inv_lt[inv_lt.norm == r.norm]
    ra2 = pd.to_numeric(c.get("gal_ra"), errors="coerce"); de2 = pd.to_numeric(c.get("gal_dec"), errors="coerce")
    sep = float(sep_arcsec(r.ra_deg, r.dec_deg, ra2.iloc[0], de2.iloc[0])) if len(c) and pd.notna(ra2.iloc[0]) else np.nan
    pos_rows.append({"norm": r.norm, "in_corpus": len(c) > 0, "sep_table1_vs_corpus_arcsec": sep})
pos = pd.DataFrame(pos_rows)
record_check("Matching", "every CDS LITTLE THINGS galaxy found in the corpus by name", bool(pos.in_corpus.all()),
             pos.loc[~pos.in_corpus, "norm"].tolist() or "all found", severity="warning")
far = pos[pos.sep_table1_vs_corpus_arcsec > POS_TOL_ARCSEC]
record_check("Matching", f"CDS and corpus LITTLE THINGS positions agree within {POS_TOL_ARCSEC:.0f} arcsec", far.empty,
             far.values.tolist()[:10] or f"max {pos.sep_table1_vs_corpus_arcsec.max():.1f} arcsec", severity="warning")
matches_new = matches_new.merge(pos.assign(survey="LITTLE_THINGS"), on=["norm", "survey"], how="left")

# Updated match file: all Phase 1 rows preserved + new per-galaxy rows
p1_keep = p1_dups.copy(); p1_keep.insert(0, "row_source", "phase1_pair (unchanged)")
mn = matches_new.copy(); mn.insert(0, "row_source", "01B_extension_vs_SPARC")
matches_updated = pd.concat([p1_keep, mn], ignore_index=True, sort=False)
display(matches_new.groupby(["survey", "duplicate_status"]).size().to_frame("n_galaxies"))

INFO: [PASS] Matching | every CDS LITTLE THINGS galaxy found in the corpus by name | all found


INFO:rar_01b:[PASS] Matching | every CDS LITTLE THINGS galaxy found in the corpus by name | all found


INFO: [PASS] Matching | CDS and corpus LITTLE THINGS positions agree within 30 arcsec | max 0.0 arcsec


INFO:rar_01b:[PASS] Matching | CDS and corpus LITTLE THINGS positions agree within 30 arcsec | max 0.0 arcsec


n_galaxies
survey        duplicate_status                              
LITTLE_THINGS ambiguous_possible_sparc_duplicate           3
              confirmed_sparc_duplicate                    3
              independent                                 20
THINGS        ambiguous_possible_sparc_duplicate           1
              confirmed_sparc_duplicate                   14
              independent                                 19

## 10. Recalculate Feasibility Categories and the Gate

A galaxy can only reach **A** if gas and stellar-disk velocity profiles (and bulge where applicable) are `verified_numeric` and aligned with the observed radii, in addition to the observed curve, uncertainties, distance, inclination and M/L information. Only **A and independent** galaxies count toward the gate of 15.

In [13]:
REQUIRED_FOR_A = ["radius", "observed_rotation_velocity", "velocity_uncertainty", "gas_velocity_contribution",
                  "stellar_disk_velocity_contribution", "bulge_velocity_contribution", "distance", "inclination",
                  "stellar_mass_to_light", "radii_compatibility"]
OK_FOR_A = {"verified_numeric", "not_applicable"}

def new_category(row):
    if row["category_from_files"] == "C":
        return "C"
    st = {q: row[f"status_{q}"] for q in REQUIRED_FOR_A}
    return "A" if all(s in OK_FOR_A for s in st.values()) else "B"

p1_feas = pd.read_csv(P1["feasibility"]); p1_feas["norm"] = p1_feas["galaxy"].map(norm_name)
audits = pd.concat([lt_audit.assign(survey="LITTLE_THINGS"), things_audit.assign(survey="THINGS")], ignore_index=True, sort=False)
audits["category_01B"] = audits.apply(new_category, axis=1)
audits["duplicate_status"] = [dup_status.get((n, s), "independent") for n, s in zip(audits.norm, audits.survey)]
audits["independent_of_sparc"] = audits.duplicate_status == "independent"
audits["missing_for_A"] = audits.apply(lambda r: "; ".join(q for q in REQUIRED_FOR_A if r[f"status_{q}"] not in OK_FOR_A), axis=1)
audits["counts_toward_gate"] = (audits.category_01B == "A") & audits.independent_of_sparc

audits["galaxy_01B"] = audits["galaxy"]
feas_updated = p1_feas.merge(audits[["norm", "survey", "galaxy_01B", "category_01B", "duplicate_status", "independent_of_sparc",
                                     "missing_for_A", "counts_toward_gate", "has_numeric_baryonic_decomposition"]],
                             on=["norm", "survey"], how="outer", suffixes=("_phase1", "_01B"), indicator=True)
feas_updated = feas_updated.rename(columns={"feasibility_category": "category_phase1"})
feas_updated["category_changed"] = feas_updated.category_phase1 != feas_updated.category_01B
feas_updated["record_presence"] = feas_updated["_merge"].map({"both": "phase1 and 01B", "left_only": "phase1 only",
                                                               "right_only": "01B only (e.g. CDS galaxy not in corpus)"})
feas_updated = feas_updated.drop(columns="_merge")
feas_updated["galaxy"] = feas_updated["galaxy"].fillna(feas_updated["galaxy_01B"])
wal = feas_updated.survey == "WALLABY"
feas_updated.loc[wal, "category_01B"] = feas_updated.loc[wal, "category_phase1"]
feas_updated.loc[wal, "counts_toward_gate_01B"] = False
feas_updated.loc[wal, "missing_for_A"] = "WALLABY excluded from main analysis (proposal Section 3); not re-assessed"

ext = audits
counts = ext.category_01B.value_counts().reindex(["A", "B", "C"], fill_value=0)
N_EXT = len(ext)
N_WITH_DECOMP = int(ext.has_numeric_baryonic_decomposition.sum())
N_CONFIRMED_DUP = int(ext.duplicate_status.str.startswith("confirmed").sum())
N_AMBIG = int((ext.duplicate_status == "ambiguous_possible_sparc_duplicate").sum())
N_INDEP = int(ext.independent_of_sparc.sum())
N_READY_INDEP = int(ext.counts_toward_gate.sum())
N_NEED_BARY = int(((ext.category_01B == "B") & ext.missing_for_A.str.contains("velocity_contribution")).sum())
GATE_PASSES = N_READY_INDEP >= GATE_MIN_INDEPENDENT
print(f"External galaxies investigated: {N_EXT} (THINGS {int((ext.survey=='THINGS').sum())}, LITTLE THINGS {int((ext.survey=='LITTLE_THINGS').sum())})")
print(f"With verified numerical baryonic decompositions: {N_WITH_DECOMP}")
print(f"Categories (01B): A={counts['A']}, B={counts['B']}, C={counts['C']}")
print(f"Confirmed SPARC duplicates: {N_CONFIRMED_DUP}; ambiguous: {N_AMBIG}; independent: {N_INDEP}")
print(f"Category-B galaxies still requiring baryonic components: {N_NEED_BARY}")
print(f"Ready AND independent: {N_READY_INDEP} vs threshold {GATE_MIN_INDEPENDENT} -> GATE {'PASSES' if GATE_PASSES else 'NOT PASSED'}")
display(ext.groupby(["survey", "category_01B", "duplicate_status"]).size().to_frame("n"))

External galaxies investigated: 60 (THINGS 34, LITTLE THINGS 26)
With verified numerical baryonic decompositions: 0
Categories (01B): A=0, B=45, C=15
Confirmed SPARC duplicates: 17; ambiguous: 4; independent: 39
Category-B galaxies still requiring baryonic components: 45
Ready AND independent: 0 vs threshold 15 -> GATE NOT PASSED


n
survey        category_01B duplicate_status                      
LITTLE_THINGS B            ambiguous_possible_sparc_duplicate   3
                           confirmed_sparc_duplicate            3
                           independent                         20
THINGS        B            confirmed_sparc_duplicate           13
                           independent                          6
              C            ambiguous_possible_sparc_duplicate   1
                           confirmed_sparc_duplicate            1
                           independent                         13

## 11. Missing Items and Source Registry

In [14]:
NEXT_SOURCE = {
    "LITTLE_THINGS": ("Oh et al. 2015 (AJ 149, 180) mass-model components are not in CDS J/AJ/149/180. Request the gas and "
                      "stellar rotation-curve components (with radii, units and the Upsilon/IMF assumptions) from the authors, "
                      "or locate a documented public release; then place the files in 01_Datasets/LITTLE_THINGS/ with a source note."),
    "THINGS": ("de Blok et al. 2008 (AJ 136, 2648) mass-model components were not located in a public machine-readable "
               "archive. Request component tables from the authors (the GitHub repository in the registry states that its "
               "THINGS curves came from W.J.G. de Blok; verify contents and licence before use)."),
}
miss = components_inventory[~components_inventory.status.isin(["verified_numeric", "not_applicable"])].copy()
miss = miss.merge(audits[["galaxy", "survey", "duplicate_status"]], on=["galaxy", "survey"], how="left")
miss["blocks_category_A"] = miss.quantity.isin(REQUIRED_FOR_A)
miss["suggested_next_step"] = miss.survey.map(NEXT_SOURCE)
missing_items = miss

registry = pd.DataFrame([
    {"source_id": "LT_CDS", "survey": "LITTLE_THINGS", "url": SOURCES["LT_CDS"], "type": "CDS/VizieR catalogue J/AJ/149/180",
     "files_read_by_notebook": ", ".join(p.name for p in LT_FILES.values()),
     "provides": "table1: positions, distance, inclination; table2: global mass-model results (total Mgas, Mstar, halo parameters); "
                 "rotdmbar: scaled total rotation curves (Data) and halo-model curves (Model)",
     "baryonic_velocity_profiles": "no (verified from ReadMe and data in this notebook)",
     "other_files_in_catalogue": ", ".join(f for f in file_summary.file if f.lower() != "readme" and not (LT_DIR / f).exists()),
     "verification": "ReadMe parsed; files checksummed; Astropy and independent readers compared"},
    {"source_id": "LT_PAPER", "survey": "LITTLE_THINGS", "url": SOURCES["LT_PAPER"], "type": "journal article (arXiv)",
     "files_read_by_notebook": "none", "provides": "mass models (figures), methods for gas and stellar components",
     "baryonic_velocity_profiles": "described; not provided as numerical tables in the CDS catalogue",
     "other_files_in_catalogue": "", "verification": "consulted by the notebook author, 2026-10-22"},
    {"source_id": "THINGS_RC", "survey": "THINGS", "url": SOURCES["THINGS_RC"], "type": "journal article (arXiv)",
     "files_read_by_notebook": "none", "provides": "rotation curves and mass models (figures); Upsilon_3.6 and halo parameters (tables)",
     "baryonic_velocity_profiles": "not located in a public machine-readable form", "other_files_in_catalogue": "",
     "verification": "consulted by the notebook author, 2026-10-22; no CDS catalogue for AJ 136, 2648 located"},
    {"source_id": "THINGS_SURVEY_VIZIER", "survey": "THINGS", "url": SOURCES["THINGS_SURVEY_VIZIER"], "type": "VizieR J/AJ/136/2563 (Walter et al. 2008)",
     "files_read_by_notebook": "none", "provides": "survey-level galaxy properties", "baryonic_velocity_profiles": "no",
     "other_files_in_catalogue": "", "verification": "registry record consulted, 2026-10-22"},
    {"source_id": "GITHUB_THINGS_CANDIDATE", "survey": "THINGS", "url": SOURCES["GITHUB_THINGS_CANDIDATE"], "type": "third-party repository",
     "files_read_by_notebook": "none", "provides": "states it hosts THINGS rotation curves provided by W.J.G. de Blok",
     "baryonic_velocity_profiles": "unverified (contents, columns and licence not documented)", "other_files_in_catalogue": "",
     "verification": "NOT verified; do not use without author confirmation and documented columns"},
    {"source_id": "SPARC", "survey": "SPARC", "url": SOURCES["SPARC"], "type": "SPARC database",
     "files_read_by_notebook": "via Phase 1 sparc_galaxy_audit.csv", "provides": "full decompositions for SPARC galaxies, including some THINGS/LITTLE THINGS galaxies",
     "baryonic_velocity_profiles": "yes, but those galaxies are SPARC duplicates and not independent", "other_files_in_catalogue": "",
     "verification": "Phase 1 checksums"},
    {"source_id": "CORPUS", "survey": "ALL", "url": SOURCES["CORPUS"], "type": "Zenodo record (unified corpus)",
     "files_read_by_notebook": f"{UNIFIED_JSON.name} (LITTLE THINGS comparison only), Phase 1 inventory",
     "provides": "observed curves; no non-SPARC baryonic components (Phase 1)", "baryonic_velocity_profiles": "SPARC only",
     "other_files_in_catalogue": "", "verification": "Phase 1 audit"},
])
display(registry[["source_id", "survey", "baryonic_velocity_profiles", "verification"]])

,source_id,survey,baryonic_velocity_profiles,verification
0,LT_CDS,LITTLE_THINGS,no (verified from ReadMe and data in this notebook),ReadMe parsed; files checksummed; Astropy and independent readers compared
1,LT_PAPER,LITTLE_THINGS,described; not provided as numerical tables in the CDS catalogue,"consulted by the notebook author, 2026-10-22"
2,THINGS_RC,THINGS,not located in a public machine-readable form,"consulted by the notebook author, 2026-10-22; no CDS catalogue for AJ 136, 2..."
3,THINGS_SURVEY_VIZIER,THINGS,no,"registry record consulted, 2026-10-22"
4,GITHUB_THINGS_CANDIDATE,THINGS,"unverified (contents, columns and licence not documented)",NOT verified; do not use without author confirmation and documented columns
5,SPARC,SPARC,"yes, but those galaxies are SPARC duplicates and not independent",Phase 1 checksums
6,CORPUS,ALL,SPARC only,Phase 1 audit


## 12. Integrity Checks and Exports

In [15]:
# Final integrity tests
record_check("Integrity", "category B or C never counted toward the gate", not audits.loc[audits.category_01B != "A", "counts_toward_gate"].any(), "ok")
record_check("Integrity", "duplicates never counted as independent",
             not audits.loc[audits.duplicate_status != "independent", "counts_toward_gate"].any(), "ok")
record_check("Integrity", "no galaxy upgraded to A without verified numeric gas and disk profiles",
             not ((audits.category_01B == "A") & ~(audits.status_gas_velocity_contribution.eq("verified_numeric")
                                                   & audits.status_stellar_disk_velocity_contribution.eq("verified_numeric"))).any(), "ok")
record_check("Integrity", "every external galaxy has a category", audits.category_01B.isin(["A", "B", "C"]).all(), f"{len(audits)} galaxies")
record_check("Integrity", "Phase 1 duplicate rows preserved", int((matches_updated.row_source == "phase1_pair (unchanged)").sum()) == len(p1_dups),
             f"{len(p1_dups)} rows")
changed_inputs = [r.relative_path for r in file_manifest.itertuples() if r.sha256 and sha256_of(PROJECT_ROOT / r.relative_path) != r.sha256]
record_check("Integrity", "input files unchanged during the run", not changed_inputs, changed_inputs or "all re-hashed")
existing_p1_names = {p.name for p in P1.values()} | {"sparc_rar_points_baseline.csv", "sparc_rar_galaxies_baseline.csv"}
record_check("Integrity", "no output overwrites a Notebook 01/02 file", not ({p.name for p in OUT.values()} & existing_p1_names), "ok")
if counts["A"] == 0:
    logger.warning("UNRESOLVED: no THINGS or LITTLE THINGS galaxy has numerical gas and stellar velocity profiles in the files available.")

def save(df, path, reason):
    (df if df is not None and len(df) else pd.DataFrame({"status": ["NOT_GENERATED"], "reason": [reason]})).to_csv(path, index=False)

save(file_manifest, OUT["manifest"], "no files")
save(things_audit, OUT["things"], "no THINGS records in the Phase 1 inventory")
save(lt_audit, OUT["lt"], "LITTLE THINGS tables could not be read")
save(components_inventory, OUT["inventory"], "no external galaxies")
save(matches_updated, OUT["matches"], "no matches")
save(feas_updated, OUT["feas"], "no feasibility records")
save(missing_items, OUT["missing"], "no missing items")
save(registry, OUT["registry"], "no sources")
save(col_defs, OUT["lt_columns"], "ReadMe could not be parsed")
save(lt_rc_out, OUT["lt_rc"], "rotdmbar could not be read")
checks = pd.DataFrame(CHECKS); checks.to_csv(OUT["checks"], index=False)
print(checks.status.value_counts().to_string())
display(checks[checks.status != "PASS"])

INFO: [PASS] Integrity | category B or C never counted toward the gate | ok


INFO:rar_01b:[PASS] Integrity | category B or C never counted toward the gate | ok


INFO: [PASS] Integrity | duplicates never counted as independent | ok


INFO:rar_01b:[PASS] Integrity | duplicates never counted as independent | ok


INFO: [PASS] Integrity | no galaxy upgraded to A without verified numeric gas and disk profiles | ok


...

INFO:rar_01b:[PASS] Integrity | input files unchanged during the run | all re-hashed


INFO: [PASS] Integrity | no output overwrites a Notebook 01/02 file | ok


INFO:rar_01b:[PASS] Integrity | no output overwrites a Notebook 01/02 file | ok


status
PASS    45
WARN     5


,section,check,status,severity,detail
10,LT files,table2.dat: Astropy CDS reader succeeded,WARN,warning,"[""Can't find table table2.dat in /content/drive/MyDrive/RAR_Universality_Res..."
32,LT values,inclination uncertainties >= 1 deg,WARN,warning,16 galaxies with e_i < 1 deg (likely fixed in the tilted-ring fit; not a rea...
35,LT scaling,rotdmbar R0.3 equals table2 R0.3 (to table precision 0.01 kpc),WARN,warning,max |diff| 3.4400 kpc
38,LT content,Data rows have positive uncertainties,WARN,warning,1 rows without positive e_V
40,Corpus provenance,corpus LITTLE THINGS points contain no CDS halo-Model rows,WARN,warning,"25 galaxies have corpus points matching only Model rows: ['CVnIdwA', 'DDO_21..."


## 13. Report and Final Feasibility Summary

In [16]:
n_fail, n_warn = int((checks.status == "FAIL").sum()), int((checks.status == "WARN").sum())
def md_table(d, maxrows=40):
    if d is None or len(d) == 0: return "_none_\n"
    d = d.head(maxrows).astype(str).replace({"nan": "", "None": ""})
    return "| " + " | ".join(d.columns) + " |\n|" + "---|" * len(d.columns) + "\n" + "".join("| " + " | ".join(x.replace("|", "/") for x in r) + " |\n" for r in d.values)

cat_table = ext.groupby(["survey", "category_01B"]).size().unstack(fill_value=0).reset_index()
dup_table = ext.groupby(["survey", "duplicate_status"]).size().unstack(fill_value=0).reset_index()
avail = components_inventory.groupby(["survey", "quantity", "status"]).size().unstack(fill_value=0).reset_index()
corpus_note = ""
if len(corpus_cmp) and "corpus_pts_matching_model_only" in corpus_cmp:
    nm = int((corpus_cmp.corpus_pts_matching_model_only.fillna(0) > 0).sum())
    corpus_note = (f"* Corpus provenance: {nm} of {len(corpus_cmp)} corpus LITTLE THINGS galaxies have points that coincide with CDS "
                   "**halo-Model** rows. Use the CDS Data rows for LITTLE THINGS rotation curves." if nm else
                   "* Corpus provenance: corpus LITTLE THINGS points match CDS Data rows only.")

report = f'''# External Baryonic Feasibility Report — Notebook 01B

*Generated by `01B_External_Baryonic_Data_Audit.ipynb` on {RUN_TIMESTAMP} (UTC). This file exists only if the notebook ran.*

## Headline
* External galaxies investigated: **{N_EXT}** (THINGS {int((ext.survey=="THINGS").sum())}, LITTLE THINGS {int((ext.survey=="LITTLE_THINGS").sum())}).
* With verified numerical baryonic decompositions: **{N_WITH_DECOMP}**.
* Categories after 01B: **A = {counts["A"]}, B = {counts["B"]}, C = {counts["C"]}**.
* SPARC duplicates: {N_CONFIRMED_DUP} confirmed, {N_AMBIG} ambiguous; independent galaxies: {N_INDEP}.
* Ready and independent: **{N_READY_INDEP}** vs threshold {GATE_MIN_INDEPENDENT}. **Cross-survey feasibility gate: {"PASSES" if GATE_PASSES else "NOT PASSED"}.**
* Integrity checks: {len(checks)} run, {n_fail} FAIL, {n_warn} WARN.

## 1. Files actually read
{md_table(file_manifest[["survey", "role", "file", "size_bytes", "status"]])}
THINGS folder: **{THINGS_FOLDER_STATE}**.

## 2. What the LITTLE THINGS files contain (from the CDS ReadMe and the data)
* `table1.dat`: positions, distance (Mpc), inclination and its error (deg), and other galaxy properties.
* `table2.dat`: global mass-model results, including total gas and stellar **masses** and halo parameters. These are not radial profiles.
* `rotdmbar.dat`: rotation curves scaled by R0.3 and V0.3; `Type` = {sorted(types.unique())}. Data rows are the observed total curve; Model rows are a fitted halo curve. Converted to kpc and km/s in `little_things_rotation_curves_physical_units.csv` (Data and Model kept separate).
* Gas, stellar-disk or bulge **velocity-profile** columns found: **{"none" if component_candidates.empty else component_candidates.label.tolist()}**.
* Other catalogue files listed in the ReadMe but not present locally: {", ".join(not_local) or "none"} (none of these is described as a gas or stellar velocity profile; check the ReadMe text before relying on that).
{corpus_note}

## 3. Availability of required quantities
{md_table(avail)}

## 4. Categories
{md_table(cat_table)}
Upgraded to A: **{", ".join(ext.loc[ext.category_01B == "A", "galaxy"]) or "none"}**.

## 5. SPARC duplicates
{md_table(dup_table)}
Full list: `03_Processed_Data/cross_survey_matches_updated.csv` (Phase 1 rows preserved).

## 6. Gate decision
**{"PASSES" if GATE_PASSES else "NOT PASSED"}**: {N_READY_INDEP} ready, independent galaxies (threshold {GATE_MIN_INDEPENDENT}). {"" if GATE_PASSES else "Following the proposal, the analysis proceeds SPARC-only (RQ1, RQ2, RQ4) and RQ3 is reported as a feasibility result unless new verified component data are added."}

## 7. What to obtain next
* **LITTLE THINGS:** {NEXT_SOURCE["LITTLE_THINGS"]}
* **THINGS:** {NEXT_SOURCE["THINGS"]}
* Per-galaxy list: `03_Processed_Data/external_data_missing_items.csv`. Sources: `03_Processed_Data/external_data_source_registry.csv`.

## 8. Checks needing attention
{md_table(checks[checks.status != "PASS"][["section", "check", "status", "detail"]])}
'''
REPORT_PATH.write_text(report, encoding="utf-8")

print("=" * 72); print("NOTEBOOK 01B — EXTERNAL BARYONIC FEASIBILITY SUMMARY"); print("=" * 72)
print(f"External galaxies investigated:            {N_EXT}")
print(f"With usable baryonic decompositions:       {N_WITH_DECOMP}")
print(f"Independent (not SPARC duplicates):        {N_INDEP}")
print(f"Categories A / B / C:                       {counts['A']} / {counts['B']} / {counts['C']}")
print(f"SPARC duplicates confirmed / ambiguous:    {N_CONFIRMED_DUP} / {N_AMBIG}")
print(f"Still requiring baryonic components:       {N_NEED_BARY}")
print(f"Feasibility gate (>= {GATE_MIN_INDEPENDENT} ready & independent): {'PASS' if GATE_PASSES else 'NOT PASSED'} ({N_READY_INDEP})")
print("Saved outputs:")
for k, p in OUT.items():
    print("   ", p)
print("    ", REPORT_PATH); print("    ", LOG_PATH)
if not GATE_PASSES:
    print("Remaining requirement: numerical gas and stellar-disk velocity profiles (with radii, uncertainties and Upsilon "
          "assumptions) for THINGS and LITTLE THINGS galaxies; see external_data_missing_items.csv.")
if n_fail:
    raise RuntimeError(f"{n_fail} critical check(s) failed:\n" + checks[checks.status == "FAIL"][["check", "detail"]].to_string(index=False))
print("\nNotebook 01B complete.")

NOTEBOOK 01B — EXTERNAL BARYONIC FEASIBILITY SUMMARY
External galaxies investigated:            60
With usable baryonic decompositions:       0
Independent (not SPARC duplicates):        39
Categories A / B / C:                       0 / 45 / 15
SPARC duplicates confirmed / ambiguous:    17 / 4
Still requiring baryonic components:       45
Feasibility gate (>= 15 ready & independent): NOT PASSED (0)
Saved outputs:
    /content/drive/MyDrive/RAR_Universality_Research/03_Processed_Data/external_baryonic_file_manifest.csv
    /content/drive/MyDrive/RAR_Universality_Research/03_Processed_Data/things_baryonic_audit.csv
    /content/drive/MyDrive/RAR_Universality_Research/03_Processed_Data/little_things_baryonic_audit.csv
    /content/drive/MyDrive/RAR_Universality_Research/03_Processed_Data/external_baryonic_components_inventory.csv
    /content/drive/MyDrive/RAR_Universality_Research/03_Processed_Data/cross_survey_matches_updated.csv
    /content/drive/MyDrive/RAR_Universality_Research/03_